<img src="https://upload.wikimedia.org/wikipedia/commons/3/35/Uba_fiuba_ingenieria_logo.png" width="300" align="center">



# **Analisis de Series de Tiempo II**

# **Clase 2, Feature Engineering**

## **2.1 El "cuando"**

¿En qué punto del tiempo estoy parado? No miramos el valor de la serie, solo la fecha. Lunes o domingo, enero o julio, feriado o día común. Capturamos la estacionalidad, los patrones que se repiten según dónde caemos en el ciclo (semana, año), sin necesidad de saber qué pasó antes.

Importamos lo necesario

In [ ]:
import numpy as np
import pandas as pd

### **A) Dataset**

Definimos serie de juguete: 14 dias diarios

In [ ]:
fechas = pd.date_range("2024-12-23", periods=14, freq="D")  # 14 fechas diarias consecutivas
y = [98, 95, 45, 92, 96, 55, 50, 97, 94, 42, 93, 95, 52, 48] # ventas inventadas (bajan finde/feriado)
df = pd.DataFrame({"y": y}, index=fechas)

### **B) Calendar Features**

Todas derivadas del indice de fechas

In [ ]:
df["dia_semana"] = df.index.dayofweek # 0 = lunes ... 6 = domingo
df["nombre_dia"] = df.index.day_name() # nombre legible del dia (solo para ver)
df["mes"] = df.index.month  # numero de mes (1... 12)
df["dia_mes"] = df.index.day # dia del mes (1... 31)
df["trimestre"] = df.index.quarter # trimestre (1... 4)
df["es_finde"] = (df.index.dayofweek >= 5).astype(int) # 1 si es sabado/domingo, 0 si no

### **C) Holidays / Eventos**

In [ ]:
feriados = pd.to_datetime(["2024-12-25", "2025-01-01"]) # lista de feriados (Navidad, Anio Nuevo)
df["es_feriado"] = df.index.isin(feriados).astype(int) # feature binaria: 1 si la fecha es feriado
df["dias_al_feriado"] = [min(abs((d - f).days) for f in feriados) # distancia (en dias) al feriado mas cercano
                         for d in df.index] # captura el build-up antes y la resaca despues

print(df[["y","nombre_dia","es_finde","es_feriado","dias_al_feriado"]].to_string())  # muestra la tabla de features

             y nombre_dia  es_finde  es_feriado  dias_al_feriado
2024-12-23  98     Monday         0           0                2
2024-12-24  95    Tuesday         0           0                1
2024-12-25  45  Wednesday         0           1                0
2024-12-26  92   Thursday         0           0                1
2024-12-27  96     Friday         0           0                2
2024-12-28  55   Saturday         1           0                3
2024-12-29  50     Sunday         1           0                3
2024-12-30  97     Monday         0           0                2
2024-12-31  94    Tuesday         0           0                1
2025-01-01  42  Wednesday         0           1                0
2025-01-02  93   Thursday         0           0                1
2025-01-03  95     Friday         0           0                2
2025-01-04  52   Saturday         1           0                3
2025-01-05  48     Sunday         1           0                4


### **D) Encoding cíclico**



¿Por qué dos componentes (sin y cos)?


Problema: 23h y 0h son vecinas, pero como entero estan a 23.

In [ ]:
horas = np.arange(24) # las 24 horas del dia: 0,1,...,23
sin_h = np.sin(2 * np.pi * horas / 24) # componente seno de cada hora
cos_h = np.cos(2 * np.pi * horas / 24) # componente coseno de cada hora

In [ ]:
d_entero = abs(23 - 0) # distancia entre 23h y 0h tratadas como enteros
d_ciclica = np.hypot(sin_h[23]-sin_h[0], cos_h[23]-cos_h[0])  # distancia real entre 23h y 0h en el circulo
d_adyacente = np.hypot(sin_h[1]-sin_h[0],  cos_h[1]-cos_h[0])   # distancia entre 0h y 1h (referencia)

In [ ]:
print("23h vs 0h, como entero :", d_entero) # da 23 (el modelo las cree lejanas)
print("23h vs 0h, ciclica :", round(d_ciclica, 3)) # da 0.261 (en el circulo estan pegadas)
print("0h  vs 1h, ciclica :", round(d_adyacente, 3))  # da 0.261 (misma distancia que 23h a 0h)

23h vs 0h, como entero : 23
23h vs 0h, ciclica : 0.261
0h  vs 1h, ciclica : 0.261


---
## **2.2 El "qué paso"**

¿Qué venía pasando con la serie hasta recién? Acá sí miramos los valores: cuánto fue ayer (lag), cuánto fue el promedio de la última semana (rolling), qué tan tembloroso venía (rolling std).

Definimos una serie de juguete con 10 valores

In [ ]:
s = pd.Series([10, 12, 13, 12, 15, 16, 14, 18, 20, 19], # 10 valores inventados
              index=pd.date_range("2024-01-01", periods=10, freq="D"),  # fechas diarias como indice
              name="y")  # nombre de la serie
df = pd.DataFrame({"y": s})  # tabla donde iremos agregando features

### **D) Lags**


Traer el pasado como columna (la "memoria" del modelo)

In [ ]:
df["lag_1"] = df["y"].shift(1) # valor de ayer (t-1)
df["lag_2"] = df["y"].shift(2) # valor de anteayer (t-2)

### **E) Rolling**

Resumen de los ultimos k valores

* REGLA: .shift(1) al final, la ventana no incluye el presente (sin leakage)


In [ ]:
df["roll_mean_3"] = df["y"].rolling(3).mean().shift(1) # media de los 3 dias previos (nivel local)
df["roll_std_3"] = df["y"].rolling(3).std().shift(1) # desvio de los 3 previos (volatilidad local)
df["roll_max_3"] = df["y"].rolling(3).max().shift(1) # maximo de los 3 previos (techo reciente)
df["roll_min_3"] = df["y"].rolling(3).min().shift(1) # minimo de los 3 previos (piso reciente)

### **F) Expanding**

Acumula todo el historial previo (ventana que crece)


In [ ]:
df["exp_mean"] = df["y"].expanding().mean().shift(1) # media de todo lo anterior a t

In [ ]:
print(df.round(2).to_string()) # muestra la tabla (NaN al inicio = falta de historial)

             y  lag_1  lag_2  roll_mean_3  roll_std_3  roll_max_3  roll_min_3  exp_mean
2024-01-01  10    NaN    NaN          NaN         NaN         NaN         NaN       NaN
2024-01-02  12   10.0    NaN          NaN         NaN         NaN         NaN     10.00
2024-01-03  13   12.0   10.0          NaN         NaN         NaN         NaN     11.00
2024-01-04  12   13.0   12.0        11.67        1.53        13.0        10.0     11.67
2024-01-05  15   12.0   13.0        12.33        0.58        13.0        12.0     11.75
2024-01-06  16   15.0   12.0        13.33        1.53        15.0        12.0     12.40
2024-01-07  14   16.0   15.0        14.33        2.08        16.0        12.0     13.00
2024-01-08  18   14.0   16.0        15.00        1.00        16.0        14.0     13.14
2024-01-09  20   18.0   14.0        16.00        2.00        18.0        14.0     13.75
2024-01-10  19   20.0   18.0        17.33        3.06        20.0        14.0     14.44
